In [14]:
# Install required packages
# pip install -q pandas scikit-learn==1.2.2 numpy xgboost

# Import libraries
import pandas as pd
import numpy as np
import os
import shutil
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional

In [15]:
def preprocess(
    train_df_with_labels: pd.DataFrame,
    test_features_df: Optional[pd.DataFrame] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Preprocesses data for the competition.
    """
    train_targets = train_df_with_labels[["timeDiff", "status"]]
    train_features = train_df_with_labels.drop(columns=["timeDiff", "status"])
    cols_to_drop = ["id", "user", "pool", "Index Event", "Outcome Event", "type", "timestamp"]
    train_features = train_features.drop(columns=cols_to_drop, errors="ignore")
    categorical_cols = train_features.select_dtypes(include=["object", "category"]).columns
    for col in categorical_cols:
        top_categories = train_features[col].value_counts().nlargest(10).index
        train_features[col] = train_features[col].where(train_features[col].isin(top_categories), "Other")
    train_features_encoded = pd.get_dummies(train_features, columns=categorical_cols, dummy_na=True, drop_first=True)
    numerical_cols = train_features_encoded.select_dtypes(include=np.number).columns
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features_encoded[numerical_cols])
    train_features_final = pd.DataFrame(train_features_scaled, index=train_features_encoded.index, columns=numerical_cols).fillna(0)
    cols_to_keep = train_features_final.columns[train_features_final.var() != 0]
    train_features_final = train_features_final[cols_to_keep]
    test_processed_features = None
    if test_features_df is not None:
        test_features = test_features_df.drop(columns=cols_to_drop, errors="ignore")
        for col in categorical_cols:
            top_categories = train_features[col].value_counts().nlargest(10).index
            test_features[col] = test_features[col].where(test_features[col].isin(top_categories), "Other")
        test_features_encoded = pd.get_dummies(test_features, columns=categorical_cols, dummy_na=True, drop_first=True)
        train_cols = train_features_encoded.columns
        test_features_aligned = test_features_encoded.reindex(columns=train_cols, fill_value=0)
        test_features_scaled = scaler.transform(test_features_aligned[numerical_cols])
        test_features_final = pd.DataFrame(test_features_scaled, index=test_features_aligned.index, columns=numerical_cols).fillna(0)
        test_processed_features = test_features_final[cols_to_keep]
    return train_features_final, train_targets, test_processed_features

In [16]:
# Define path to the single participant data folder.
PARTICIPANT_DATA_PATH = '/home/dgxuser40/manjil/finsurv/participant_data'
SUBMISSION_DIR = 'test'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

In [17]:
# Define all 16 event pairs
index_events = ["Deposit", "Borrow", "Repay", "Withdraw"]
outcome_events = index_events + ["Liquidated"]
event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing and Predicting for: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- Load and Preprocess ---
    try:
        train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
        # The test features for participants are the validation features.
        test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))
    except FileNotFoundError as e:
        print(f"Data not found for {dataset_path}. Skipping.")
        continue
        
    X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

    # --- Train Model ---
    try:
        # For XGBoost survival:cox, we need to create labels in the format:
        # negative values for censored, positive for events
        # The absolute value represents the time
        # y_train_xgb = np.where(
        #     y_train['status'] == 1,
        #     y_train['timeDiff'],  # Positive for events
        #     -y_train['timeDiff']  # Negative for censored
        # )

        y_train_xgb = y_train['timeDiff'].values 
        
        # Create DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_train, label=y_train_xgb)
        dtest = xgb.DMatrix(X_test_processed)
        
        # XGBoost parameters for survival analysis
        params = {
            'objective': 'survival:cox',
            'eval_metric': 'cox-nloglik',
            'max_depth': 3,
            'eta': 0.1,  # learning rate
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 5,
            'lambda': 1.0,  # L2 regularization
            'alpha': 0.0,   # L1 regularization
            'seed': 42
        }
        
        print(f"Training XGBoost survival:cox model for {dataset_path}...")
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=100,
            verbose_eval=False
        )
        print("  - Model trained successfully.")

        # --- Generate and Save Predictions ---
        print(f"Generating predictions for {dataset_path}...")
        # Predict risk scores (higher score = higher risk)
        predictions = -model.predict(dtest)
        
        # Save predictions to a CSV file
        prediction_filename = dataset_path.replace(os.sep, '_') + '.csv'
        prediction_save_path = os.path.join(SUBMISSION_DIR, prediction_filename)
        pd.DataFrame(predictions).to_csv(prediction_save_path, header=False, index=False)
        print(f"  -> Predictions saved to {prediction_save_path}")
        
    except Exception as e:
        print(f"\nERROR: The model for {dataset_path} failed to train.")
        print(f"Details: {e}")

print("\n\nAll prediction files have been generated.")


Processing and Predicting for: Deposit -> Borrow
Training XGBoost survival:cox model for Deposit/Borrow...
  - Model trained successfully.
Generating predictions for Deposit/Borrow...
  -> Predictions saved to test/Deposit_Borrow.csv

Processing and Predicting for: Deposit -> Repay


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
model.print_summary()

NameError: name 'model' is not defined

In [ ]:
output_zip_filename = 'submission_xgb'
shutil.make_archive(output_zip_filename, 'zip', SUBMISSION_DIR)

print(f"Successfully created '{output_zip_filename}.zip' from the '{SUBMISSION_DIR}' directory.")
print("You can now upload this file to the CodaBench competition.")

Successfully created 'submission_xgb.zip' from the '../my_prediction_submission_xgb' directory.
You can now upload this file to the CodaBench competition.
